# House Prices: Advanced Regression Techniques
## Leakage-Free Machine Learning Pipeline with Out-of-Fold Evaluation

### Project Overview

This project predicts residential sale prices using the Ames Housing dataset from Kaggle's **House Prices: Advanced Regression Techniques** competition.

The main objective is not only to obtain a competitive score, but to build an evaluation procedure that estimates performance on unseen data as honestly as possible.

The project therefore focuses on:

- leakage-free preprocessing;
- true out-of-fold (OOF) predictions;
- explicit hypotheses before experiments;
- interpretation of results after experiments;
- transparent structural outlier analysis;
- comparison of several regression models under the same validation procedure;
- ensemble selection based on OOF evidence;
- detailed error analysis;
- comparison between internal validation and the Kaggle public leaderboard.

The experimental workflow is:

\[
\text{hypothesis}
\rightarrow
\text{experiment}
\rightarrow
\text{OOF evaluation}
\rightarrow
\text{error analysis}
\rightarrow
\text{decision}
\]

A central rule throughout the notebook is that **learned preprocessing must never be fitted on validation-fold or Kaggle test information**.


## 1. Environment and Libraries

The project uses `pandas` and `NumPy` for data manipulation, `scikit-learn` for preprocessing, pipelines and cross-validation, and gradient-boosting libraries for nonlinear models.

A fixed `random_state` is used where applicable to make the experiments reproducible.


In [1]:
# Cell 1 - Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_log_error

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge

import xgboost as xgb
import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

print("Imports loaded successfully.")

Imports loaded successfully.


## 2. Problem Formulation

The task is to predict the sale price of a house based on explanatory
variables describing its physical, qualitative, temporal, and
location-related characteristics.

Let each house be represented by a feature vector

$$
\mathbf{x}_i =
(x_{i1}, x_{i2}, \ldots, x_{ip})
$$

where:

- $\mathbf{x}_i$ is the feature vector for the $i$-th house,
- $p$ is the number of explanatory variables,
- $y_i$ is the true sale price of the $i$-th house.

The goal is to learn a regression function

$$
f(\mathbf{x}_i) \approx y_i
$$

that generalizes well to previously unseen houses.

Because house prices are positive and strongly right-skewed, the
target variable is transformed using

$$
z_i = \log(1+y_i).
$$

The models are trained to predict $z_i$ rather than the raw sale price.

After prediction, the inverse transformation is applied:

$$
\hat{y}_i = \exp(\hat{z}_i)-1.
$$

The logarithmic transformation reduces target skewness and limits the
influence of very large absolute price differences. It is also directly
aligned with the RMSLE evaluation metric used by the Kaggle competition.

At this stage, the raw training and test datasets are loaded without
fitting any preprocessing operation. All learned preprocessing steps
will later be fitted exclusively inside the training folds of
cross-validation to prevent data leakage.

In [2]:
# Cell 2 - Load raw data

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)

train.head()

Train shape: (1460, 81)
Test shape : (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 3. Mathematical Background

### 3.1 Root Mean Squared Logarithmic Error (RMSLE)

The Kaggle competition evaluates predictions using the
Root Mean Squared Logarithmic Error:

$$
\mathrm{RMSLE}
=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
\left(
\log(1+y_i)
-
\log(1+\hat{y}_i)
\right)^2
}.
$$

where:

- $n$ is the number of observations,
- $y_i$ is the actual sale price,
- $\hat{y}_i$ is the predicted sale price.

### Why RMSLE?

RMSLE is appropriate for this problem because:

1. House prices have a strongly right-skewed distribution.
2. Errors are evaluated in logarithmic rather than raw price space.
3. Very expensive houses have less influence than they would under
   ordinary RMSE on raw prices.
4. It is the official Kaggle evaluation metric.

Since this project trains the models on

$$
z_i=\log(1+y_i),
$$

the validation error can be calculated directly in transformed target
space:

$$
\mathrm{RMSE}_{log}
=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
(\hat{z}_i-z_i)^2
}.
$$

For positive prices, this corresponds to RMSLE after applying the
inverse transformation.

### 3.2 Leakage-Free Cross-Validation

For each cross-validation fold, the training data are separated into

$$
D_{train}^{(k)}
\quad\text{and}\quad
D_{valid}^{(k)}.
$$

The preprocessing transformation is fitted only on the training fold:

$$
T_k = \mathrm{fit}(D_{train}^{(k)}).
$$

The same fitted transformation is then applied to the validation fold:

$$
X_{valid}^{(k)*}
=
T_k(X_{valid}^{(k)}).
$$

Therefore, information from the validation fold cannot influence
imputation, scaling, or categorical encoding.

This prevents data leakage and produces genuine out-of-fold predictions.

### 3.3 Target and Predictor Separation

Before any learned preprocessing is performed, the target variable
`SalePrice` is separated from the predictors.

The training target is transformed to logarithmic space using

$$
z_i = \log(1+y_i).
$$

Therefore:

- `X` contains only the training predictors;
- `y` contains `log1p(SalePrice)`;
- `X_test` contains the untouched Kaggle test predictors.

No imputer, scaler, or categorical encoder is fitted at this stage.
These operations will be performed later inside the cross-validation
pipeline to prevent data leakage.

In [3]:
# Cell 3 - Separate features and target

X = train.drop(columns=["SalePrice"]).copy()
y = np.log1p(train["SalePrice"])

X_test = test.copy()

print("X shape     :", X.shape)
print("y shape     :", y.shape)
print("X_test shape:", X_test.shape)

print("\nTarget (log1p SalePrice):")
print(y.describe())

X shape     : (1460, 80)
y shape     : (1460,)
X_test shape: (1459, 80)

Target (log1p SalePrice):
count    1460.000000
mean       12.024057
std         0.399449
min        10.460271
25%        11.775105
50%        12.001512
75%        12.273736
max        13.534474
Name: SalePrice, dtype: float64


## 4. Leakage-Safe Feature Engineering

### Hypothesis

Domain-inspired features may represent property size, age, amenities and quality
more directly than the original variables.

The engineered variables use **predictor columns only**; `SalePrice` is not used.
Therefore, the transformations themselves do not learn information from the target.

Examples include total floor area, total bathrooms, building/remodelling age,
porch area, amenity indicators and quality-size interactions.

For example, total structural floor area is defined as

$$
\mathrm{TotalSF}
=
\mathrm{TotalBsmtSF}
+
\mathrm{1stFlrSF}
+
\mathrm{2ndFlrSF}.
$$

The bathroom feature is defined as

$$
\mathrm{TotalBath}
=
\mathrm{FullBath}
+
0.5\,\mathrm{HalfBath}
+
\mathrm{BsmtFullBath}
+
0.5\,\mathrm{BsmtHalfBath}.
$$

Two interaction features combine overall quality with property size:

$$
\mathrm{Qual\_TotalSF}
=
\mathrm{OverallQual}\times\mathrm{TotalSF},
$$

$$
\mathrm{Qual\_GrLivArea}
=
\mathrm{OverallQual}\times\mathrm{GrLivArea}.
$$

### Missing Values During Feature Engineering

Missing values are replaced by zero only when zero has a direct structural
meaning for the engineered quantity (for example, no basement area or no
garage area).

No statistics such as means, medians, modes, or category frequencies are
learned from the full dataset during feature engineering.

Statistical imputation and categorical encoding are performed later inside
the cross-validation pipeline, where they are fitted only on each training fold.

In [4]:
# Cell 4 - Leakage-safe feature engineering

def add_features(df):
    data = df.copy()

    # Total living area
    data["TotalSF"] = (
        data["TotalBsmtSF"].fillna(0)
        + data["1stFlrSF"].fillna(0)
        + data["2ndFlrSF"].fillna(0)
    )

    # Total number of bathrooms
    data["TotalBath"] = (
        data["FullBath"].fillna(0)
        + 0.5 * data["HalfBath"].fillna(0)
        + data["BsmtFullBath"].fillna(0)
        + 0.5 * data["BsmtHalfBath"].fillna(0)
    )

    # Age-related features
    data["HouseAge"] = data["YrSold"] - data["YearBuilt"]
    data["RemodAge"] = data["YrSold"] - data["YearRemodAdd"]

    # Total porch area
    data["TotalPorchSF"] = (
        data["OpenPorchSF"].fillna(0)
        + data["EnclosedPorch"].fillna(0)
        + data["3SsnPorch"].fillna(0)
        + data["ScreenPorch"].fillna(0)
        + data["WoodDeckSF"].fillna(0)
    )

    # Binary features
    data["HasGarage"] = (data["GarageArea"].fillna(0) > 0).astype(int)
    data["HasBsmt"] = (data["TotalBsmtSF"].fillna(0) > 0).astype(int)
    data["HasFireplace"] = (data["Fireplaces"].fillna(0) > 0).astype(int)
    data["HasPool"] = (data["PoolArea"].fillna(0) > 0).astype(int)

    # Quality / size interactions
    data["Qual_TotalSF"] = data["OverallQual"] * data["TotalSF"]
    data["Qual_GrLivArea"] = data["OverallQual"] * data["GrLivArea"]

    return data


X_fe = add_features(X)
X_test_fe = add_features(X_test)

print("Before feature engineering:", X.shape)
print("After feature engineering :", X_fe.shape)
print("Test shape after FE        :", X_test_fe.shape)

new_features = [col for col in X_fe.columns if col not in X.columns]
print("\nNew features:")
print(new_features)


Before feature engineering: (1460, 80)
After feature engineering : (1460, 91)
Test shape after FE        : (1459, 91)

New features:
['TotalSF', 'TotalBath', 'HouseAge', 'RemodAge', 'TotalPorchSF', 'HasGarage', 'HasBsmt', 'HasFireplace', 'HasPool', 'Qual_TotalSF', 'Qual_GrLivArea']


## 5. Leakage-Free Preprocessing

This is the most important methodological safeguard in the project.

The dataset contains both numerical and categorical features, so separate
preprocessing pipelines are defined for the two groups.

For numerical variables:

- missing values are imputed using the median;
- features are standardized using `StandardScaler`.

For categorical variables:

- missing values are replaced with the explicit category `"Missing"`;
- categories are encoded using one-hot encoding;
- previously unseen categories are safely ignored during transformation.

The complete preprocessing step is defined inside a `ColumnTransformer`.

### Leakage Prevention

At this stage, the preprocessor is **defined but not fitted**.

During cross-validation, a fresh clone of the complete preprocessing pipeline
is fitted only on the training portion of each fold:

$$
T_k = \mathrm{fit}(X_{\mathrm{train}}^{(k)}).
$$

The fitted transformation is then applied to the validation portion:

$$
X_{\mathrm{valid}}^{(k)*}
=
T_k(X_{\mathrm{valid}}^{(k)}).
$$

This ensures that statistics such as numerical medians and categorical
encoding information are learned only from the corresponding training fold.

The Kaggle test set is never used to fit imputation, scaling, or encoding.

In [5]:
# Cell 5 - Leakage-free preprocessing

# Identify numerical and categorical columns
numeric_features = X_fe.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_fe.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features  :", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features      :", len(numeric_features) + len(categorical_features))


# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


# Combine numerical and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("\nLeakage-free preprocessor created successfully.")
print("IMPORTANT: The preprocessor has NOT been fitted yet.")

Numerical features  : 48
Categorical features: 43
Total features      : 91

Leakage-free preprocessor created successfully.
IMPORTANT: The preprocessor has NOT been fitted yet.


## 6. Baseline Model: Leakage-Free Ridge Cross-Validation

### Hypothesis

Ridge regression provides a strong, interpretable regularized linear baseline for a high-dimensional one-hot encoded housing dataset.

Five-fold cross-validation is used. For every fold:

1. the preprocessor is fitted only on the training fold;
2. the Ridge model is trained on that fold;
3. predictions are generated for the untouched validation fold;
4. the predictions are stored in their original row positions.

After all five folds, every training observation has exactly one prediction produced by a model that did not train on that observation. These are genuine **out-of-fold predictions**.

The reported OOF RMSLE is therefore the primary internal validation measure.

### Mathematical Definition of Out-of-Fold Predictions

For fold \(k\), the complete preprocessing and regression pipeline is fitted
using only the corresponding training subset:

$$
\hat{f}_k
=
\mathrm{fit}
\left(
X_{\mathrm{train}}^{(k)},
y_{\mathrm{train}}^{(k)}
\right).
$$

The validation observations are then predicted using this fitted pipeline:

$$
\hat{y}_i^{OOF}
=
\hat{f}_k(\mathbf{x}_i),
\qquad
i \in D_{\mathrm{valid}}^{(k)}.
$$

Therefore, every out-of-fold prediction is produced by a model that has never
seen that observation during training.

The overall OOF error is calculated as

$$
\mathrm{RMSLE}_{OOF}
=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
\left(
\hat{z}_i^{OOF}-z_i
\right)^2
}.
$$

Because the target is modeled in log-space, this quantity corresponds to
RMSLE on the original house-price scale.

In [7]:
# Cell 6 - Leakage-free 5-fold CV with Ridge

from sklearn.base import clone

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_ridge = np.zeros(len(X_fe))
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    kf.split(X_fe),
    start=1
):

    X_train_fold = X_fe.iloc[train_idx]
    X_valid_fold = X_fe.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    # Fresh pipeline for every fold
    fold_pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", Ridge(alpha=10.0))
    ])

    # Preprocessing and model fitting happen
    # ONLY on the training fold
    fold_pipeline.fit(
        X_train_fold,
        y_train_fold
    )

    # Validation fold is only transformed and predicted
    valid_pred = fold_pipeline.predict(
        X_valid_fold
    )

    oof_ridge[valid_idx] = valid_pred

    fold_rmse = np.sqrt(
        np.mean(
            (valid_pred - y_valid_fold) ** 2
        )
    )

    fold_scores.append(fold_rmse)

    print(
        f"Fold {fold} RMSLE: "
        f"{fold_rmse:.5f}"
    )


overall_rmsle = np.sqrt(
    np.mean(
        (oof_ridge - y) ** 2
    )
)

print("\n------------------------------")
print(
    f"Mean Fold RMSLE : "
    f"{np.mean(fold_scores):.5f}"
)
print(
    f"Std Fold RMSLE  : "
    f"{np.std(fold_scores):.5f}"
)
print(
    f"OOF RMSLE       : "
    f"{overall_rmsle:.5f}"
)
print("------------------------------")

Fold 1 RMSLE: 0.13315
Fold 2 RMSLE: 0.12910
Fold 3 RMSLE: 0.22172
Fold 4 RMSLE: 0.12300
Fold 5 RMSLE: 0.10674

------------------------------
Mean Fold RMSLE : 0.14274
Std Fold RMSLE  : 0.04050
OOF RMSLE       : 0.14838
------------------------------


### Interpretation

The leakage-free Ridge baseline achieves an overall OOF RMSLE of **0.14838**.

The fold scores vary substantially, with a standard deviation of **0.04050**.
In particular, Fold 3 produces an RMSLE of **0.22172**, considerably worse
than the remaining folds.

This suggests that the overall error may be strongly influenced by a small
number of difficult or structurally unusual observations.

Instead of removing observations immediately, the next step performs
observation-level error analysis to identify which houses generate the
largest genuine OOF errors and to investigate why.

This distinction is important: outliers are investigated based on model
behavior and property characteristics before any exclusion rule is defined.

## 7. OOF Error Analysis

A single RMSLE value does not explain *why* a model fails.

The OOF residuals are inspected observation by observation. This is important because unusually large errors can reveal rare property types, data-quality issues, nonlinear relationships or influential observations that destabilize cross-validation.

For each observation, the OOF residual in log-price space is defined as

$$
r_i = z_i - \hat{z}_i^{OOF},
$$

and its absolute error is

$$
|r_i| = \left|z_i - \hat{z}_i^{OOF}\right|.
$$

Large values of $|r_i|$ identify observations for which the model produces
unusually inaccurate predictions without having trained on those observations.

In [8]:
# Cell 7 - OOF error analysis

error_analysis = pd.DataFrame({
    "Id": train["Id"],
    "ActualLogPrice": y,
    "PredictedLogPrice": oof_ridge
})

error_analysis["Residual"] = (
    error_analysis["ActualLogPrice"]
    - error_analysis["PredictedLogPrice"]
)

error_analysis["AbsError"] = np.abs(
    error_analysis["Residual"]
)

# Convert log predictions back to original price scale
error_analysis["ActualPrice"] = np.expm1(
    error_analysis["ActualLogPrice"]
)

error_analysis["PredictedPrice"] = np.expm1(
    error_analysis["PredictedLogPrice"]
)

# Largest OOF errors
worst_errors = error_analysis.nlargest(
    15,
    "AbsError"
)

print("Top 15 largest OOF prediction errors:\n")

display(
    worst_errors[
        [
            "Id",
            "ActualPrice",
            "PredictedPrice",
            "Residual",
            "AbsError"
        ]
    ]
)

Top 15 largest OOF prediction errors:



,Id,ActualPrice,PredictedPrice,Residual,AbsError
1298,1299,160000.0,2.662808e+06,-2.811957,2.811957
523,524,184750.0,1.076036e+06,-1.762031,1.762031
632,633,82500.0,1.762849e+05,-0.759297,0.759297
30,31,40000.0,8.545037e+04,-0.759043,0.759043
1182,1183,745000.0,3.774395e+05,0.679973,0.679973
462,463,62383.0,1.229676e+05,-0.678621,0.678621
1324,1325,147000.0,2.808474e+05,-0.647376,0.647376
495,496,34900.0,6.634935e+04,-0.642434,0.642434
968,969,37900.0,7.109272e+04,-0.629022,0.629022
812,813,55993.0,9.715282e+04,-0.551051,0.551051


### Interpretation

The observation-level analysis reveals two exceptionally large OOF errors.

For **Id 1299**, the actual sale price is approximately \$160,000, while the
OOF prediction is approximately \$2.66 million, producing an absolute
log-space error of **2.812**.

For **Id 524**, the actual sale price is approximately \$184,750, while the
OOF prediction is approximately \$1.08 million, producing an absolute
log-space error of **1.762**.

These errors are substantially larger than those of the remaining
observations. The next-largest absolute log error is approximately **0.759**.

However, large prediction errors alone are **not sufficient justification
for removing observations**. The next step therefore investigates the
physical, qualitative, location and sale characteristics of these houses
before defining any structural outlier rule.

## 8. Inspecting the Largest OOF Errors

The largest absolute log-errors are joined back to meaningful raw housing attributes and to their validation-fold assignment.

The purpose is diagnostic rather than automatic deletion: a large model error alone is **not** sufficient justification for removing an observation.

The fold assignment is inspected to determine whether the poor baseline
performance is caused by a general modeling weakness or concentrated in a
small number of influential validation observations.

In [9]:
# Cell 8 - Inspect the worst OOF cases and their folds

# Assign each observation to its validation fold
fold_assignment = np.zeros(len(X_fe), dtype=int)

for fold, (_, valid_idx) in enumerate(kf.split(X_fe), start=1):
    fold_assignment[valid_idx] = fold

error_analysis["Fold"] = fold_assignment

# Important original features for inspection
inspection_cols = [
    "Id",
    "SalePrice",
    "GrLivArea",
    "OverallQual",
    "OverallCond",
    "YearBuilt",
    "YearRemodAdd",
    "Neighborhood",
    "LotArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "GarageCars",
    "GarageArea",
    "SaleCondition"
]

inspection = train[inspection_cols].copy()

inspection["Fold"] = fold_assignment
inspection["OOF_PredictedPrice"] = np.expm1(oof_ridge)
inspection["AbsLogError"] = error_analysis["AbsError"]

worst_inspection = inspection.nlargest(
    15,
    "AbsLogError"
)

display(worst_inspection)

,Id,SalePrice,GrLivArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,Neighborhood,LotArea,TotalBsmtSF,1stFlrSF,2ndFlrSF,GarageCars,GarageArea,SaleCondition,Fold,OOF_PredictedPrice,AbsLogError
1298,1299,160000,5642,10,5,2008,2008,Edwards,63887,6110,4692,950,2,1418,Partial,3,2.662808e+06,2.811957
523,524,184750,4676,10,5,2007,2008,Edwards,40094,3138,3138,1538,3,884,Partial,3,1.076036e+06,1.762031
632,633,82500,1411,7,5,1977,1977,NWAmes,11900,1386,1411,0,2,544,Family,5,1.762849e+05,0.759297
30,31,40000,1317,4,4,1920,1950,IDOTRR,8500,649,649,668,1,250,Normal,1,8.545037e+04,0.759043
1182,1183,745000,4476,10,5,1996,1996,NoRidge,15623,2396,2411,2065,3,813,Abnorml,2,3.774395e+05,0.679973
462,463,62383,864,5,5,1965,1965,Sawyer,8281,864,864,0,1,360,Normal,2,1.229676e+05,0.678621
1324,1325,147000,1795,8,5,2006,2007,Somerst,9986,1795,1795,0,3,895,Partial,3,2.808474e+05,0.647376
495,496,34900,720,4,5,1920,1950,IDOTRR,7879,720,720,0,0,0,Abnorml,4,6.634935e+04,0.642434
968,969,37900,968,3,6,1910,1950,OldTown,5925,600,600,368,0,0,Abnorml,4,7.109272e+04,0.629022
812,813,55993,1044,5,5,1952,1952,IDOTRR,8712,540,1044,0,2,504,Alloca,1,9.715282e+04,0.551051


### Interpretation

The two largest OOF errors, Id 1299 and Id 524, both belong to Fold 3.

This explains why Fold 3 is substantially worse than the remaining folds.
However, the fact that both observations are difficult for the model is still
not sufficient reason to remove them.

The next step therefore measures their influence on the global OOF RMSLE and
then compares their structural characteristics with the distribution of the
full training dataset.

## 9. Sensitivity to the Two Most Extreme Cases

This diagnostic calculation measures how strongly the two largest errors affect the global OOF RMSLE.

It is not yet an outlier-removal decision. The observations must first be investigated structurally and a reproducible rule must be defined independently of their validation residuals.


In [10]:
# Cell 9 - Quantify the influence of the two extreme observations

extreme_ids = [524, 1299]

extreme_mask = train["Id"].isin(extreme_ids)
normal_mask = ~extreme_mask

# OOF RMSLE on all observations
rmsle_all = np.sqrt(
    np.mean((oof_ridge - y) ** 2)
)

# Same OOF predictions, but evaluate without the two extreme observations
rmsle_without_extremes = np.sqrt(
    np.mean(
        (oof_ridge[normal_mask] - y[normal_mask]) ** 2
    )
)

print(f"OOF RMSLE - all observations       : {rmsle_all:.5f}")
print(f"OOF RMSLE - excluding Id 524/1299 : {rmsle_without_extremes:.5f}")

print(
    f"Difference                        : "
    f"{rmsle_all - rmsle_without_extremes:.5f}"
)

print("\nExtreme observations:")
display(
    error_analysis.loc[
        extreme_mask,
        [
            "Id",
            "ActualPrice",
            "PredictedPrice",
            "Fold",
            "AbsError"
        ]
    ]
)

OOF RMSLE - all observations       : 0.14838
OOF RMSLE - excluding Id 524/1299 : 0.12039
Difference                        : 0.02799

Extreme observations:


,Id,ActualPrice,PredictedPrice,Fold,AbsError
523,524,184750.0,1.076036e+06,3,1.762031
1298,1299,160000.0,2.662808e+06,3,2.811957


### Interpretation

Removing the two observations only from the OOF evaluation reduces the
measured RMSLE from **0.14838** to **0.12039**, a difference of **0.02799**.

This demonstrates that the baseline validation score is highly sensitive to
these two observations.

However, **0.12039 is not a retrained cross-validation score**. The OOF
predictions were generated using the original training folds, and the two
observations were excluded only when recalculating the diagnostic metric.

Therefore, this result is used only as evidence of influence. A legitimate
outlier-removal experiment requires a structural rule based on predictor
characteristics, followed by complete leakage-free cross-validation on the
resulting training dataset.

## 10. Structural Investigation of Extreme Observations

The two extreme cases are compared with dataset percentiles and relevant housing characteristics.

The question is whether these observations represent an unusual relationship between property characteristics and sale price, rather than merely being inconvenient predictions for the model.


In [11]:
# Cell 10 - Investigate the two extreme observations

key_features = [
    "GrLivArea",
    "LotArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "GarageArea",
    "OverallQual",
    "SalePrice"
]

print("Dataset percentiles:")
display(
    train[key_features].quantile(
        [0.50, 0.90, 0.95, 0.99, 1.00]
    )
)

print("\nExtreme observations:")
display(
    train.loc[
        train["Id"].isin([524, 1299]),
        [
            "Id",
            "Neighborhood",
            "GrLivArea",
            "LotArea",
            "TotalBsmtSF",
            "1stFlrSF",
            "2ndFlrSF",
            "GarageArea",
            "OverallQual",
            "OverallCond",
            "SaleType",
            "SaleCondition",
            "SalePrice"
        ]
    ]
)

Dataset percentiles:


,GrLivArea,LotArea,TotalBsmtSF,1stFlrSF,2ndFlrSF,GarageArea,OverallQual,SalePrice
0.50,1464.00,9478.50,991.50,1087.00,0.00,480.00,6.0,163000.00
0.90,2158.30,14381.70,1602.20,1680.00,954.20,757.10,8.0,278000.00
0.95,2466.10,17401.15,1753.00,1831.25,1141.05,850.10,8.0,326100.00
0.99,3123.48,37567.64,2155.05,2219.46,1418.92,1002.79,10.0,442567.01
1.00,5642.00,215245.00,6110.00,4692.00,2065.00,1418.00,10.0,755000.00



Extreme observations:


,Id,Neighborhood,GrLivArea,LotArea,TotalBsmtSF,1stFlrSF,2ndFlrSF,GarageArea,OverallQual,OverallCond,SaleType,SaleCondition,SalePrice
523,524,Edwards,4676,40094,3138,3138,1538,884,10,5,New,Partial,184750
1298,1299,Edwards,5642,63887,6110,4692,950,1418,10,5,New,Partial,160000


### Interpretation

Both observations are structurally unusual relative to the training
distribution.

Id 524 has a `GrLivArea` of **4676 sq ft**, while Id 1299 has **5642 sq ft**,
compared with a 99th percentile of approximately **3123 sq ft**.

Both properties also have `OverallQual = 10`, indicating the highest overall
quality rating, and unusually large basement and first-floor areas.

Despite these characteristics, their sale prices are only **$184,750** and
**$160,000**, respectively. This creates an unusual relationship between
property size, quality and observed sale price.

Importantly, the structural evidence is based on predictor characteristics
and the observed target relationship rather than on OOF residual magnitude
alone.

The next step defines an explicit and reproducible structural rule for these
observations before retraining and re-evaluating the model.

## 11. Transparent Structural Outlier Rule

Based on the preceding structural investigation, a reproducible training-data
rule is defined:

$$
\mathrm{GrLivArea} > 4000
\quad\land\quad
\mathrm{SalePrice} < 300000.
$$

The key methodological distinction is that observations are **not removed
simply because their OOF prediction errors are large**.

Instead, the rule describes an unusual relationship between an extremely
large above-ground living area and a comparatively low observed sale price.

The rule uses the target variable and is therefore applied **only as a
training-data cleaning rule**. It is never applied to the Kaggle test set and
is not part of the inference pipeline.

The rule is stated explicitly rather than removing observations by their
row indices or `Id` values, making the data-cleaning decision transparent and
reproducible.

Before any observations are removed, all rows satisfying the rule are
displayed for inspection.


In [12]:
# Cell 11 - Define a transparent structural outlier rule

# Large houses with unusually low sale prices
outlier_rule = (
    (train["GrLivArea"] > 4000)
    & (train["SalePrice"] < 300000)
)

structural_outliers = train.loc[
    outlier_rule,
    [
        "Id",
        "GrLivArea",
        "OverallQual",
        "Neighborhood",
        "SaleType",
        "SaleCondition",
        "SalePrice"
    ]
]

print("Number of structural outliers:", outlier_rule.sum())
print()

display(structural_outliers)

Number of structural outliers: 2



,Id,GrLivArea,OverallQual,Neighborhood,SaleType,SaleCondition,SalePrice
523,524,4676,10,Edwards,New,Partial,184750
1298,1299,5642,10,Edwards,New,Partial,160000


### Interpretation

The structural rule identifies exactly two unusually large, low-priced
properties discovered during the diagnostic analysis.

Importantly, the observations are not removed simply because they produced
large OOF residuals. They satisfy an explicit and reproducible training-data
criterion based on the relationship between `GrLivArea` and `SalePrice`.

Because the rule uses the target variable, it is applied only to the training
data and never to the Kaggle test set.

The next experiment removes these observations and repeats the complete
leakage-free cross-validation procedure from scratch. This tests whether the
structural cleaning rule improves both predictive performance and
fold-to-fold stability using new genuine OOF predictions.


## 12. Ridge After Structural Outlier Removal

### Hypothesis

If the two structurally unusual properties are high-leverage observations, removing them should improve both average OOF performance and fold-to-fold stability.

Cross-validation is repeated **from scratch** on the cleaned training set. A fresh preprocessing/model pipeline is fitted independently inside every fold, preserving the leakage-free methodology.


In [14]:
# Cell 12 - Retrain leakage-free Ridge after structural outlier removal

# Keep observations that do NOT satisfy the structural outlier rule
clean_mask = ~outlier_rule

X_clean = X_fe.loc[clean_mask].reset_index(drop=True)
y_clean = y.loc[clean_mask].reset_index(drop=True)

print("Original training observations:", len(X_fe))
print("Removed structural outliers   :", outlier_rule.sum())
print("Clean training observations   :", len(X_clean))

# New CV from scratch
kf_clean = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_ridge_clean = np.zeros(len(X_clean))
clean_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    kf_clean.split(X_clean),
    start=1
):

    X_train_fold = X_clean.iloc[train_idx]
    X_valid_fold = X_clean.iloc[valid_idx]

    y_train_fold = y_clean.iloc[train_idx]
    y_valid_fold = y_clean.iloc[valid_idx]

    # IMPORTANT:
    # Create a fresh pipeline for every fold
    fold_pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", Ridge(alpha=10.0))
    ])

    # Preprocessing is fitted ONLY on this training fold
    fold_pipeline.fit(
        X_train_fold,
        y_train_fold
    )

    valid_pred = fold_pipeline.predict(
        X_valid_fold
    )

    oof_ridge_clean[valid_idx] = valid_pred

    fold_rmse = np.sqrt(
        np.mean(
            (valid_pred - y_valid_fold) ** 2
        )
    )

    clean_fold_scores.append(fold_rmse)

    print(
        f"Fold {fold} RMSLE: "
        f"{fold_rmse:.5f}"
    )


clean_oof_rmsle = np.sqrt(
    np.mean(
        (oof_ridge_clean - y_clean) ** 2
    )
)

print("\n----------------------------------")
print(
    f"Mean Fold RMSLE : "
    f"{np.mean(clean_fold_scores):.5f}"
)
print(
    f"Std Fold RMSLE  : "
    f"{np.std(clean_fold_scores):.5f}"
)
print(
    f"OOF RMSLE       : "
    f"{clean_oof_rmsle:.5f}"
)
print("----------------------------------")

Original training observations: 1460
Removed structural outliers   : 2
Clean training observations   : 1458
Fold 1 RMSLE: 0.12305
Fold 2 RMSLE: 0.11048
Fold 3 RMSLE: 0.11705
Fold 4 RMSLE: 0.12355
Fold 5 RMSLE: 0.10185

----------------------------------
Mean Fold RMSLE : 0.11520
Std Fold RMSLE  : 0.00819
OOF RMSLE       : 0.11549
----------------------------------


### Interpretation

After applying the structural training-data cleaning rule and repeating the
complete leakage-free cross-validation procedure from scratch, Ridge improves
from an OOF RMSLE of **0.14838** to **0.11549**.

Fold-to-fold variability also decreases substantially:

$$
\sigma_{\mathrm{fold}}
:
0.04050
\rightarrow
0.00819.
$$

The previously weak Fold 3 improves from **0.22172** to **0.11705**, while the
remaining folds also produce consistently competitive scores.

The reduction in both overall OOF error and fold-to-fold variance supports the
hypothesis that the two structurally unusual observations had disproportionate
influence on the Ridge model.

Importantly, **0.11549 is a genuine retrained OOF result**, unlike the earlier
diagnostic value of 0.12039. All preprocessing and model fitting were repeated
independently inside each training fold after applying the explicit structural
training-data cleaning rule.

## 13. Experiment Summary

The experiment table records not only scores but also the interpretation of each change.

This prevents the project from becoming a sequence of unexplained model versions: every experiment should answer a specific question and lead to an explicit decision.


In [15]:
# Cell 13 - Experiment summary

experiment_results = pd.DataFrame({
    "Experiment": [
        "Ridge - all observations",
        "Ridge - diagnostic exclusion only",
        "Ridge - retrained after structural outlier rule"
    ],
    "OOF_RMSLE": [
        rmsle_all,
        rmsle_without_extremes,
        clean_oof_rmsle
    ],
    "Interpretation": [
        "Leakage-free baseline; strongly affected by two extreme observations",
        "Diagnostic only - observations excluded from scoring, no retraining",
        "Full leakage-free CV retrained on 1458 observations"
    ]
})

display(experiment_results)

,Experiment,OOF_RMSLE,Interpretation
0,Ridge - all observations,0.148376,Leakage-free baseline; strongly affected by tw...
1,Ridge - diagnostic exclusion only,0.120386,Diagnostic only - observations excluded from s...
2,Ridge - retrained after structural outlier rule,0.115489,Full leakage-free CV retrained on 1458 observa...


## 14. XGBoost Under the Same Validation Protocol

### Hypothesis

House prices contain nonlinear effects and feature interactions that a linear Ridge model may not capture. XGBoost is therefore evaluated as a nonlinear gradient-boosted tree model.

For a fair comparison, it uses the same cleaned observations, the same five folds and the same fold-specific preprocessing procedure as Ridge.


In [17]:
# Cell 14 - Leakage-free 5-fold CV with XGBoost

xgb_oof = np.zeros(len(X_clean))
xgb_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    kf_clean.split(X_clean),
    start=1
):
    X_train_fold = X_clean.iloc[train_idx]
    X_valid_fold = X_clean.iloc[valid_idx]

    y_train_fold = y_clean.iloc[train_idx]
    y_valid_fold = y_clean.iloc[valid_idx]

    # Fresh pipeline for every fold
    xgb_pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),

        ("model", xgb.XGBRegressor(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=3,
            min_child_weight=1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ))
    ])

    # Preprocessing + model are fitted ONLY on training fold
    xgb_pipeline.fit(
        X_train_fold,
        y_train_fold
    )

    valid_pred = xgb_pipeline.predict(
        X_valid_fold
    )

    xgb_oof[valid_idx] = valid_pred

    fold_rmse = np.sqrt(
        np.mean(
            (valid_pred - y_valid_fold) ** 2
        )
    )

    xgb_fold_scores.append(fold_rmse)

    print(
        f"Fold {fold} RMSLE: "
        f"{fold_rmse:.5f}"
    )


xgb_oof_rmsle = np.sqrt(
    np.mean(
        (xgb_oof - y_clean) ** 2
    )
)

print("\n----------------------------------")
print(
    f"Mean Fold RMSLE : "
    f"{np.mean(xgb_fold_scores):.5f}"
)
print(
    f"Std Fold RMSLE  : "
    f"{np.std(xgb_fold_scores):.5f}"
)
print(
    f"OOF RMSLE       : "
    f"{xgb_oof_rmsle:.5f}"
)
print("----------------------------------")

Fold 1 RMSLE: 0.11827
Fold 2 RMSLE: 0.10966
Fold 3 RMSLE: 0.12039
Fold 4 RMSLE: 0.12158
Fold 5 RMSLE: 0.10612

----------------------------------
Mean Fold RMSLE : 0.11520
Std Fold RMSLE  : 0.00617
OOF RMSLE       : 0.11537
----------------------------------


### Interpretation

XGBoost achieves an OOF RMSLE of **0.11537**, compared with **0.11549**
for Ridge after the same structural training-data cleaning rule.

The improvement in overall OOF error is small, but XGBoost also shows lower
fold-to-fold variability:

$$
\sigma_{\mathrm{fold}}:
0.00819
\rightarrow
0.00617.
$$

This suggests that the nonlinear boosted-tree model is slightly more stable
than Ridge under the same five validation folds.

However, the standalone improvement is modest. Therefore, XGBoost is not
selected solely because it has a marginally lower OOF score.

The next step compares the OOF prediction errors of Ridge and XGBoost to test
whether the two model families capture complementary structure. If their
errors are sufficiently different, an ensemble may generalize better than
either model alone.

## 15. Systematic XGBoost tuning with Optuna

### Hypothesis

The leakage-free XGBoost baseline may improve through controlled hyperparameter optimization. Instead of testing many arbitrary model variants, Optuna searches a compact, interpretable parameter space.

The optimization problem is

\[
\theta^*
=
\arg\min_{\theta}
\mathrm{RMSLE}_{OOF}(\theta),
\]

where \(\theta\) is the XGBoost hyperparameter vector.

### Leakage safeguard

Optuna does **not** receive preprocessed full-dataset features. Every trial repeats the same five-fold procedure used by the baseline. Inside every fold, a fresh clone of the preprocessor is fitted only on the training portion. Therefore the validation fold cannot influence imputation, scaling, categorical encoding or model fitting.

The number of trials is deliberately limited. The purpose is to test a clear hypothesis, not to turn the notebook into an opaque AutoML search.


In [ ]:
# Cell 15A - Optuna leakage-free XGBoost tuning
# Optuna - leakage-free XGBoost tuning
# Install once if necessary: pip install optuna

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def optuna_xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 700, 1300, step=200),
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.045),
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 4),
        "subsample": trial.suggest_float("subsample", 0.65, 0.90),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.90),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 0.20),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 3.0),
    }

    trial_oof = np.zeros(len(X_clean))

    for train_idx, valid_idx in kf_clean.split(X_clean):
        X_train_fold = X_clean.iloc[train_idx]
        X_valid_fold = X_clean.iloc[valid_idx]
        y_train_fold = y_clean.iloc[train_idx]

        trial_pipeline = Pipeline(steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", xgb.XGBRegressor(
                **params,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            ))
        ])

        trial_pipeline.fit(X_train_fold, y_train_fold)
        trial_oof[valid_idx] = trial_pipeline.predict(X_valid_fold)

    return np.sqrt(np.mean((trial_oof - y_clean) ** 2))

study = optuna.create_study(direction="minimize")
study.optimize(optuna_xgb_objective, n_trials=15)

print("Best Optuna OOF RMSLE:", f"{study.best_value:.6f}")
print("Best parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

baseline_xgb_score = xgb_oof_rmsle
print("\nBaseline XGBoost OOF RMSLE:", f"{baseline_xgb_score:.6f}")
print("Optuna difference          :", f"{study.best_value - baseline_xgb_score:+.6f}")


Best Optuna OOF RMSLE: 0.113803
Best parameters:
  n_estimators: 1100
  learning_rate: 0.039524534394525634
  max_depth: 3
  min_child_weight: 3
  subsample: 0.7750590251178121
  colsample_bytree: 0.6515493505666603
  reg_alpha: 0.1030285029296229
  reg_lambda: 0.6672670420130714

Baseline XGBoost OOF RMSLE: 0.115370
Optuna difference          : -0.001566


### Validation of the Best Optuna Configuration

The Optuna study identifies the hyperparameter configuration with the lowest
OOF objective value among the tested trials.

To inspect the selected configuration more carefully, the best parameters are
evaluated again using the same five leakage-free folds.

For every fold:

1. a fresh clone of the preprocessing pipeline is created;
2. preprocessing is fitted only on the training subset;
3. XGBoost is trained only on the same training subset;
4. predictions are generated only for the untouched validation subset.

This produces a complete set of tuned OOF predictions and allows both overall
RMSLE and fold-to-fold variability to be inspected.

The tuned model is accepted only if the improvement is reproducible and the
fold scores remain reasonably stable.


In [19]:
# Cell 15B - Validate the best Optuna XGBoost configuration

optuna_xgb_oof = np.zeros(len(X_clean))
optuna_xgb_fold_scores = []
best_xgb_params = study.best_params.copy()

for fold, (train_idx, valid_idx) in enumerate(
    kf_clean.split(X_clean),
    start=1
):
    X_train_fold = X_clean.iloc[train_idx]
    X_valid_fold = X_clean.iloc[valid_idx]
    y_train_fold = y_clean.iloc[train_idx]
    y_valid_fold = y_clean.iloc[valid_idx]

    tuned_pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", xgb.XGBRegressor(
            **best_xgb_params,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ))
    ])

    tuned_pipeline.fit(X_train_fold, y_train_fold)
    valid_pred = tuned_pipeline.predict(X_valid_fold)
    optuna_xgb_oof[valid_idx] = valid_pred

    fold_rmsle = np.sqrt(
        np.mean((valid_pred - y_valid_fold) ** 2)
    )
    optuna_xgb_fold_scores.append(fold_rmsle)

    print(f"Fold {fold} RMSLE: {fold_rmsle:.5f}")

optuna_xgb_oof_rmsle = np.sqrt(
    np.mean((optuna_xgb_oof - y_clean) ** 2)
)

print("\n----------------------------------")
print(f"Mean Fold RMSLE : {np.mean(optuna_xgb_fold_scores):.5f}")
print(f"Std Fold RMSLE  : {np.std(optuna_xgb_fold_scores):.5f}")
print(f"OOF RMSLE       : {optuna_xgb_oof_rmsle:.5f}")
print("----------------------------------")

print("\nBASELINE vs OPTUNA")
print("------------------")
print(f"Baseline XGBoost : {xgb_oof_rmsle:.6f}")
print(f"Optuna XGBoost   : {optuna_xgb_oof_rmsle:.6f}")
print(f"Difference       : {optuna_xgb_oof_rmsle - xgb_oof_rmsle:+.6f}")


Fold 1 RMSLE: 0.11697
Fold 2 RMSLE: 0.10977
Fold 3 RMSLE: 0.11618
Fold 4 RMSLE: 0.11988
Fold 5 RMSLE: 0.10562

----------------------------------
Mean Fold RMSLE : 0.11368
Std Fold RMSLE  : 0.00521
OOF RMSLE       : 0.11380
----------------------------------

BASELINE vs OPTUNA
------------------
Baseline XGBoost : 0.115370
Optuna XGBoost   : 0.113803
Difference       : -0.001566


### Optuna Validation Interpretation

The repeated OOF evaluation checks the selected Optuna configuration under the
same folds and the same leakage-free preprocessing protocol as the baseline.

A lower tuned OOF RMSLE together with reasonable fold-to-fold variability is
evidence that the tuned configuration is a better XGBoost candidate for the
next comparison and ensemble steps.

This keeps model selection evidence-based rather than assuming that the lowest
Optuna trial value must automatically be adopted.


## 16. Recording Baseline and Tuned XGBoost Results

Both the baseline and Optuna-tuned XGBoost models are added to the same
experiment table.

This makes the effect of hyperparameter tuning explicit and avoids opaque
version labels such as v1, v2, or v3.


In [20]:
# Cell 16 - Record baseline and tuned XGBoost results

# Prevent duplicate XGBoost rows if this cell is re-run
experiment_results = experiment_results[
    ~experiment_results["Experiment"].str.startswith("XGBoost")
].copy()

xgb_results = pd.DataFrame({
    "Experiment": [
        "XGBoost - baseline leakage-free CV",
        "XGBoost - Optuna tuned leakage-free CV"
    ],
    "OOF_RMSLE": [
        xgb_oof_rmsle,
        optuna_xgb_oof_rmsle
    ],
    "Interpretation": [
        "Nonlinear baseline evaluated under the same cleaned 5-fold protocol",
        "Optuna tuning re-evaluated under the same leakage-free OOF protocol"
    ]
})

experiment_results = pd.concat(
    [experiment_results, xgb_results],
    ignore_index=True
)

display(experiment_results.sort_values("OOF_RMSLE"))


,Experiment,OOF_RMSLE,Interpretation
4,XGBoost - Optuna tuned leakage-free CV,0.113803,Optuna tuning re-evaluated under the same leak...
3,XGBoost - baseline leakage-free CV,0.115370,Nonlinear baseline evaluated under the same cl...
2,Ridge - retrained after structural outlier rule,0.115489,Full leakage-free CV retrained on 1458 observa...
1,Ridge - diagnostic exclusion only,0.120386,Diagnostic only - observations excluded from s...
0,Ridge - all observations,0.148376,Leakage-free baseline; strongly affected by tw...


### Optuna interpretation

Optuna is used only as a **proposal mechanism** for candidate hyperparameters. Model selection still depends on leakage-free OOF performance under the same folds as the baseline.

A tuned configuration should replace the baseline only when it improves OOF RMSLE while preserving reasonable fold-to-fold stability. If the gain is negligible or negative, the simpler baseline remains the preferred model. This decision rule prevents tuning complexity from being mistaken for genuine generalization improvement.


### Decision

The Optuna-tuned XGBoost configuration is retained for the subsequent
ensemble experiments.

Compared with the baseline XGBoost model, OOF RMSLE improved from
**0.115370** to **0.113803**, while fold-to-fold standard deviation
decreased from **0.00617** to **0.00521**.

Therefore, the tuned model provides both better average generalization
performance and slightly greater cross-validation stability under the
same leakage-free evaluation protocol.

## 17. Are Ridge and XGBoost Complementary?

An ensemble is useful only when its components make sufficiently different mistakes.

Two correlations are examined:

- **prediction correlation** — how similarly the models rank/predict houses;
- **error correlation** — how similarly the models fail.

Even strongly correlated predictions can produce a useful ensemble if their errors are not identical.


In [21]:
# Cell 17 - Compare Ridge and XGBoost OOF predictions

ridge_errors = y_clean - oof_ridge_clean
xgb_errors = y_clean - optuna_xgb_oof

error_correlation = np.corrcoef(
    ridge_errors,
    xgb_errors
)[0, 1]

prediction_correlation = np.corrcoef(
    oof_ridge_clean,
    optuna_xgb_oof
)[0, 1]

print(
    f"Prediction correlation : "
    f"{prediction_correlation:.5f}"
)

print(
    f"Error correlation      : "
    f"{error_correlation:.5f}"
)

print("\nIndividual OOF RMSLE:")
print(f"Ridge   : {clean_oof_rmsle:.5f}")
print(f"XGBoost (Optuna) : {optuna_xgb_oof_rmsle:.5f}")

Prediction correlation : 0.98524
Error correlation      : 0.83382

Individual OOF RMSLE:
Ridge   : 0.11549
XGBoost (Optuna) : 0.11380


## 18. Ridge + XGBoost OOF Blend

### Hypothesis

A weighted average of the linear Ridge model and nonlinear XGBoost model may reduce model-specific errors.

The blend is evaluated entirely from OOF predictions. No Kaggle leaderboard information is used to choose the internal validation score.


In [22]:
# Cell 18 - Ridge + XGBoost OOF blend

blend_results = []

# Test several Ridge/XGBoost weights
for ridge_weight in np.arange(0.0, 1.01, 0.1):

    xgb_weight = 1.0 - ridge_weight

    blend_oof = (
        ridge_weight * oof_ridge_clean
        + xgb_weight * optuna_xgb_oof
    )

    blend_rmsle = np.sqrt(
        np.mean(
            (blend_oof - y_clean) ** 2
        )
    )

    blend_results.append({
        "Ridge_weight": ridge_weight,
        "XGBoost_weight": xgb_weight,
        "OOF_RMSLE": blend_rmsle
    })


blend_results = pd.DataFrame(blend_results)

display(
    blend_results.sort_values("OOF_RMSLE")
)

best_blend = blend_results.loc[
    blend_results["OOF_RMSLE"].idxmin()
]

print("\nBest OOF blend:")
print(
    f"Ridge weight   : "
    f"{best_blend['Ridge_weight']:.1f}"
)
print(
    f"XGBoost weight : "
    f"{best_blend['XGBoost_weight']:.1f}"
)
print(
    f"OOF RMSLE      : "
    f"{best_blend['OOF_RMSLE']:.5f}"
)

,Ridge_weight,XGBoost_weight,OOF_RMSLE
5,0.5,0.5,0.109780
4,0.4,0.6,0.109803
6,0.6,0.4,0.110154
3,0.3,0.7,0.110223
7,0.7,0.3,0.110922
2,0.2,0.8,0.111036
8,0.8,0.2,0.112076
1,0.1,0.9,0.112234
9,0.9,0.1,0.113603
0,0.0,1.0,0.113803



Best OOF blend:
Ridge weight   : 0.5
XGBoost weight : 0.5
OOF RMSLE      : 0.10978


### Interpretation

The Optuna-tuned XGBoost model improves not only its standalone OOF
performance but also the Ridge-XGBoost ensemble.

The best tested blend uses equal weights:

\[
\hat{z}_{blend}
=
0.5\hat{z}_{Ridge}
+
0.5\hat{z}_{XGBoost},
\]

and achieves an OOF RMSLE of **0.109780**, compared with **0.115489**
for Ridge and **0.113803** for the tuned XGBoost model individually.

The nearby 40/60 Ridge-XGBoost blend achieves a very similar score
(**0.109803**), indicating that the improvement is not dependent on a
single sharply optimized weight.

Therefore, the 50/50 blend is retained because it provides the best
tested OOF score while remaining simple and interpretable.

## 19. Fold-by-Fold Ensemble Stability

The ensemble is evaluated separately on each validation fold.

Reporting the mean and standard deviation across folds is important: a small average error is more convincing when it is not caused by one unusually favorable split.


In [23]:
# Cell 19 - Fold-by-fold evaluation of the 50/50 OOF blend

best_blend_oof = (
    0.5 * oof_ridge_clean
    + 0.5 * optuna_xgb_oof
)

blend_fold_scores = []

for fold, (_, valid_idx) in enumerate(
    kf_clean.split(X_clean),
    start=1
):
    fold_score = np.sqrt(
        np.mean(
            (
                best_blend_oof[valid_idx]
                - y_clean.iloc[valid_idx]
            ) ** 2
        )
    )

    blend_fold_scores.append(fold_score)

    print(
        f"Fold {fold} Blend RMSLE: "
        f"{fold_score:.5f}"
    )


blend_oof_rmsle = np.sqrt(
    np.mean(
        (best_blend_oof - y_clean) ** 2
    )
)

print("\n----------------------------------")
print(f"Mean Fold RMSLE : {np.mean(blend_fold_scores):.5f}")
print(f"Std Fold RMSLE  : {np.std(blend_fold_scores):.5f}")
print(f"OOF RMSLE       : {blend_oof_rmsle:.5f}")
print("----------------------------------")

Fold 1 Blend RMSLE: 0.11607
Fold 2 Blend RMSLE: 0.10544
Fold 3 Blend RMSLE: 0.11309
Fold 4 Blend RMSLE: 0.11445
Fold 5 Blend RMSLE: 0.09887

----------------------------------
Mean Fold RMSLE : 0.10958
Std Fold RMSLE  : 0.00648
OOF RMSLE       : 0.10978
----------------------------------


### Interpretation

The 50/50 Ridge + Optuna-tuned XGBoost ensemble achieves an overall
OOF RMSLE of **0.10978** with a fold-to-fold standard deviation of
**0.00648**.

The fold scores remain relatively consistent, suggesting that the ensemble
improvement is not driven by a single unusually favorable validation split.

Compared with the individual models, the ensemble improves both predictive
accuracy and robustness across folds. Therefore, the 50/50 blend is retained
as the current best internal model.

## Experiment Tracking with MLflow

MLflow is used only as an experiment ledger. It does not participate in
feature engineering, preprocessing, cross-validation, hyperparameter
selection, or inference.

At this point the key metrics already exist, so the run can safely record:

- cleaned Ridge OOF RMSLE;
- baseline XGBoost OOF RMSLE;
- Optuna-tuned XGBoost OOF RMSLE;
- selected Ridge/XGBoost ensemble OOF RMSLE;
- the best Optuna hyperparameters.

This improves reproducibility without changing the leakage-free validation
logic.


In [24]:
# MLflow experiment tracking (auxiliary)
# MLflow - experiment tracking only
# Install once if necessary: pip install mlflow

import mlflow

mlflow.set_experiment("HousePrices_Leakage_Free")

with mlflow.start_run(run_name="Leakage_Free_Optuna_Ensemble"):
    mlflow.log_metric("ridge_clean_oof_rmsle", float(clean_oof_rmsle))
    mlflow.log_metric("xgboost_baseline_oof_rmsle", float(xgb_oof_rmsle))
    mlflow.log_metric("xgboost_optuna_oof_rmsle", float(optuna_xgb_oof_rmsle))
    mlflow.log_metric("ridge_xgb_blend_oof_rmsle", float(blend_oof_rmsle))

    for key, value in study.best_params.items():
        mlflow.log_param(f"optuna_{key}", value)

    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("validation_protocol", "leakage-free OOF")
    mlflow.log_param(
        "outlier_rule",
        "GrLivArea > 4000 and SalePrice < 300000"
    )

print("MLflow run logged successfully.")


2026/08/21 20:42:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/21 20:42:48 INFO mlflow.store.db.utils: Updating database tables
2026/08/21 20:43:11 INFO mlflow.tracking.fluent: Experiment with name 'HousePrices_Leakage_Free' does not exist. Creating a new experiment.


MLflow run logged successfully.


## 20. Ensemble Decision

The best tested Ridge/XGBoost blend is recorded in the experiment summary.

The decision is evidence-based: the ensemble is retained only if its OOF RMSLE improves on the individual models while maintaining acceptable fold stability.


In [27]:
# Cell 20 - Add the best blend to the experiment summary

# Remove previous Ridge/XGBoost blend rows if this cell is re-run
experiment_results = experiment_results[
    ~experiment_results["Experiment"].str.contains(
        "Ridge \\+.*XGBoost OOF blend",
        regex=True
    )
].copy()

blend_result = pd.DataFrame({
    "Experiment": [
        "50/50 Ridge + Optuna XGBoost OOF blend"
    ],
    "OOF_RMSLE": [
        blend_oof_rmsle
    ],
    "Interpretation": [
        "Best OOF result; tuned XGBoost improves the complementary Ridge-XGBoost ensemble"
    ]
})

experiment_results = pd.concat(
    [experiment_results, blend_result],
    ignore_index=True
)

display(
    experiment_results.sort_values("OOF_RMSLE")
)

,Experiment,OOF_RMSLE,Interpretation
5,50/50 Ridge + Optuna XGBoost OOF blend,0.109780,Best OOF result; tuned XGBoost improves the co...
4,XGBoost - Optuna tuned leakage-free CV,0.113803,Optuna tuning re-evaluated under the same leak...
3,XGBoost - baseline leakage-free CV,0.115370,Nonlinear baseline evaluated under the same cl...
2,Ridge - retrained after structural outlier rule,0.115489,Full leakage-free CV retrained on 1458 observa...
1,Ridge - diagnostic exclusion only,0.120386,Diagnostic only - observations excluded from s...
0,Ridge - all observations,0.148376,Leakage-free baseline; strongly affected by tw...


### Interpretation
Best OOF result; the Optuna-tuned XGBoost improves the complementary 50/50 Ridge-XGBoost ensemble


## 21. LightGBM Experiment

### Hypothesis

A third gradient-boosting implementation may provide additional predictive diversity.

LightGBM is tested under the same leakage-free OOF protocol. A model is not included merely because it is sophisticated; it must contribute measurable generalization value.


In [29]:
# Cell 21 - Leakage-free 5-fold CV with LightGBM

lgb_oof = np.zeros(len(X_clean))
lgb_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    kf_clean.split(X_clean),
    start=1
):
    X_train_fold = X_clean.iloc[train_idx]
    X_valid_fold = X_clean.iloc[valid_idx]

    y_train_fold = y_clean.iloc[train_idx]
    y_valid_fold = y_clean.iloc[valid_idx]

    # Fresh pipeline for every fold
    lgb_pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),

        ("model", lgb.LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=-1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        ))
    ])

    # Preprocessing and model fit ONLY on training fold
    lgb_pipeline.fit(
        X_train_fold,
        y_train_fold
    )

    valid_pred = lgb_pipeline.predict(
        X_valid_fold
    )

    lgb_oof[valid_idx] = valid_pred

    fold_rmse = np.sqrt(
        np.mean(
            (valid_pred - y_valid_fold) ** 2
        )
    )

    lgb_fold_scores.append(fold_rmse)

    print(
        f"Fold {fold} RMSLE: "
        f"{fold_rmse:.5f}"
    )


lgb_oof_rmsle = np.sqrt(
    np.mean(
        (lgb_oof - y_clean) ** 2
    )
)

print("\n----------------------------------")
print(f"Mean Fold RMSLE : {np.mean(lgb_fold_scores):.5f}")
print(f"Std Fold RMSLE  : {np.std(lgb_fold_scores):.5f}")
print(f"OOF RMSLE       : {lgb_oof_rmsle:.5f}")
print("----------------------------------")

Fold 1 RMSLE: 0.13272
Fold 2 RMSLE: 0.11759
Fold 3 RMSLE: 0.12879
Fold 4 RMSLE: 0.12758
Fold 5 RMSLE: 0.10992

----------------------------------
Mean Fold RMSLE : 0.12332
Std Fold RMSLE  : 0.00835
OOF RMSLE       : 0.12361
----------------------------------


### Interpretation

LightGBM achieves an OOF RMSLE of **0.12361**, which is weaker than both
Ridge (**0.11549**) and the Optuna-tuned XGBoost model (**0.11380**).

Its fold-to-fold standard deviation of **0.00835** is also higher than that
of the tuned XGBoost model (**0.00521**).

However, standalone performance alone is not sufficient to reject a model
from an ensemble. A weaker model can still contribute if its OOF errors are
sufficiently different from those of the stronger models.

Therefore, LightGBM is retained temporarily for an OOF error-correlation
and controlled three-model blending experiment. It will be included in the
final ensemble only if it provides measurable OOF improvement.

## 22. OOF Error-Correlation Analysis

The error correlations between Ridge, XGBoost and LightGBM are compared.

This analysis asks whether LightGBM contributes genuinely different information or largely repeats errors already made by the existing ensemble components.


In [31]:
# Cell 22 - Compare OOF error correlations
# Ridge vs Optuna-tuned XGBoost vs LightGBM

ridge_errors = y_clean - oof_ridge_clean
xgb_errors = y_clean - optuna_xgb_oof
lgb_errors = y_clean - lgb_oof

error_corr = pd.DataFrame({
    "Ridge": ridge_errors,
    "Optuna XGBoost": xgb_errors,
    "LightGBM": lgb_errors
}).corr()

print("OOF error correlation:")
display(error_corr)

print("\nIndividual OOF RMSLE:")
print(f"Ridge          : {clean_oof_rmsle:.5f}")
print(f"Optuna XGBoost : {optuna_xgb_oof_rmsle:.5f}")
print(f"LightGBM       : {lgb_oof_rmsle:.5f}")
print(f"Best 50/50 blend: {blend_oof_rmsle:.5f}")

OOF error correlation:


,Ridge,Optuna XGBoost,LightGBM
Ridge,1.000000,0.833822,0.785819
Optuna XGBoost,0.833822,1.000000,0.923493
LightGBM,0.785819,0.923493,1.000000



Individual OOF RMSLE:
Ridge          : 0.11549
Optuna XGBoost : 0.11380
LightGBM       : 0.12361
Best 50/50 blend: 0.10978


### Interpretation

The Optuna-tuned XGBoost and LightGBM models have a high OOF error
correlation of **0.9235**, indicating that the two gradient-boosting
models tend to make similar mistakes.

LightGBM is less correlated with Ridge (**0.7858**), but its standalone
OOF RMSLE (**0.12361**) is substantially worse than both Ridge
(**0.11549**) and Optuna-tuned XGBoost (**0.11380**).

In contrast, Ridge and Optuna-tuned XGBoost have a lower error correlation
of **0.8338**, which helps explain why their 50/50 ensemble improves to
an OOF RMSLE of **0.10978**.

These results suggest that LightGBM is unlikely to deserve a large ensemble
weight. However, a controlled three-model OOF blend is tested next to
determine whether a small LightGBM contribution provides any measurable
improvement.

## 23. Controlled Three-Model Blend Test

Several controlled combinations of Ridge, XGBoost and LightGBM are evaluated using OOF predictions.

This experiment tests the principle that **more models do not automatically produce a better ensemble**. Added complexity is justified only by improved validation performance.


In [33]:
# Cell 23 - Controlled three-model OOF blend test

three_model_results = []

# Test small LightGBM contributions only.
# Ridge and Optuna-tuned XGBoost share the remaining weight equally.
for lgb_weight in [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:

    remaining_weight = 1.0 - lgb_weight

    ridge_weight = remaining_weight / 2
    xgb_weight = remaining_weight / 2

    blend_3_oof = (
        ridge_weight * oof_ridge_clean
        + xgb_weight * optuna_xgb_oof
        + lgb_weight * lgb_oof
    )

    rmsle = np.sqrt(
        np.mean(
            (blend_3_oof - y_clean) ** 2
        )
    )

    three_model_results.append({
        "Ridge_weight": ridge_weight,
        "Optuna_XGBoost_weight": xgb_weight,
        "LightGBM_weight": lgb_weight,
        "OOF_RMSLE": rmsle
    })


three_model_results = pd.DataFrame(
    three_model_results
)

display(
    three_model_results.sort_values("OOF_RMSLE")
)

best_three_model = three_model_results.loc[
    three_model_results["OOF_RMSLE"].idxmin()
]

print("\nBest three-model test:")
print(
    f"Ridge          : "
    f"{best_three_model['Ridge_weight']:.2f}"
)
print(
    f"Optuna XGBoost : "
    f"{best_three_model['Optuna_XGBoost_weight']:.2f}"
)
print(
    f"LightGBM       : "
    f"{best_three_model['LightGBM_weight']:.2f}"
)
print(
    f"OOF RMSLE      : "
    f"{best_three_model['OOF_RMSLE']:.5f}"
)

print(
    f"\nCurrent 50/50 Ridge-Optuna XGBoost: "
    f"{blend_oof_rmsle:.5f}"
)

,Ridge_weight,Optuna_XGBoost_weight,LightGBM_weight,OOF_RMSLE
0,0.500,0.500,0.00,0.109780
1,0.475,0.475,0.05,0.109839
2,0.450,0.450,0.10,0.109970
3,0.425,0.425,0.15,0.110171
4,0.400,0.400,0.20,0.110442
5,0.375,0.375,0.25,0.110784
6,0.350,0.350,0.30,0.111194



Best three-model test:
Ridge          : 0.50
Optuna XGBoost : 0.50
LightGBM       : 0.00
OOF RMSLE      : 0.10978

Current 50/50 Ridge-Optuna XGBoost: 0.10978


### Interpretation

The controlled three-model experiment confirms that LightGBM does not improve
the selected ensemble.

The best result is obtained with:

- Ridge weight = **0.50**
- Optuna-tuned XGBoost weight = **0.50**
- LightGBM weight = **0.00**

with an OOF RMSLE of **0.10978**.

Every tested non-zero LightGBM weight increases the validation error.
Therefore, the additional model does not provide enough complementary
information to justify the added complexity.

The final ensemble is kept as the simpler and better-performing
50/50 Ridge + Optuna-tuned XGBoost blend.

## 24. Model-Selection Decision

The LightGBM experiment and the final model-selection decision are recorded explicitly.

A rejected model is still a useful experiment: it documents what was tested, what happened and why the final system remains simpler.


In [35]:
# Cell 24 - Add LightGBM experiments to experiment summary

best_three_model = three_model_results.sort_values(
    "OOF_RMSLE"
).iloc[0]

# Remove previous LightGBM entries if this cell is re-run
experiment_results = experiment_results[
    ~experiment_results["Experiment"].isin([
        "LightGBM - leakage-free CV after structural outlier rule",
        "Best Ridge + XGBoost + LightGBM controlled blend",
        "Best Ridge + Optuna XGBoost + LightGBM controlled blend"
    ])
].copy()

lgb_results = pd.DataFrame({
    "Experiment": [
        "LightGBM - leakage-free CV after structural outlier rule",
        "Best Ridge + Optuna XGBoost + LightGBM controlled blend"
    ],
    "OOF_RMSLE": [
        lgb_oof_rmsle,
        best_three_model["OOF_RMSLE"]
    ],
    "Interpretation": [
        "Weaker standalone model; high error correlation with Optuna-tuned XGBoost",
        "LightGBM receives zero weight; it does not improve the Ridge-Optuna XGBoost ensemble"
    ]
})

experiment_results = pd.concat(
    [experiment_results, lgb_results],
    ignore_index=True
)

display(
    experiment_results.sort_values("OOF_RMSLE")
)

,Experiment,OOF_RMSLE,Interpretation
5,50/50 Ridge + Optuna XGBoost OOF blend,0.109780,Best OOF result; tuned XGBoost improves the co...
7,Best Ridge + Optuna XGBoost + LightGBM control...,0.109780,LightGBM receives zero weight; it does not imp...
4,XGBoost - Optuna tuned leakage-free CV,0.113803,Optuna tuning re-evaluated under the same leak...
3,XGBoost - baseline leakage-free CV,0.115370,Nonlinear baseline evaluated under the same cl...
2,Ridge - retrained after structural outlier rule,0.115489,Full leakage-free CV retrained on 1458 observa...
1,Ridge - diagnostic exclusion only,0.120386,Diagnostic only - observations excluded from s...
6,LightGBM - leakage-free CV after structural ou...,0.123607,Weaker standalone model; high error correlatio...
0,Ridge - all observations,0.148376,Leakage-free baseline; strongly affected by tw...


### Interpretation

LightGBM receives zero weight in the best controlled three-model blend and
therefore does not improve the selected Ridge-Optuna XGBoost ensemble.

Every tested non-zero LightGBM contribution increases the OOF RMSLE above
the current best value of **0.10978**. Therefore, LightGBM is excluded from
the final ensemble, keeping the final model both simpler and better supported
by the validation evidence.

## 25. Final Ensemble OOF Error Analysis

After model selection, residual analysis is repeated for the selected ensemble.

This separates *model selection* from *model understanding*. The goal is to identify systematic strengths and weaknesses that a single global RMSLE score can hide.


In [36]:
# Cell 25 - Final ensemble OOF error analysis

final_error_analysis = pd.DataFrame({
    "Id": train.loc[clean_mask, "Id"].reset_index(drop=True),
    "ActualLogPrice": y_clean,
    "PredictedLogPrice": best_blend_oof
})

# Residuals in log space
final_error_analysis["Residual"] = (
    final_error_analysis["ActualLogPrice"]
    - final_error_analysis["PredictedLogPrice"]
)

final_error_analysis["AbsLogError"] = np.abs(
    final_error_analysis["Residual"]
)

# Convert predictions back to original dollar scale
final_error_analysis["ActualPrice"] = np.expm1(
    final_error_analysis["ActualLogPrice"]
)

final_error_analysis["PredictedPrice"] = np.expm1(
    final_error_analysis["PredictedLogPrice"]
)

final_error_analysis["PriceError"] = (
    final_error_analysis["PredictedPrice"]
    - final_error_analysis["ActualPrice"]
)

# Percentage error
final_error_analysis["PercentError"] = (
    final_error_analysis["PriceError"]
    / final_error_analysis["ActualPrice"]
) * 100


print("FINAL ENSEMBLE ERROR SUMMARY")
print("----------------------------")

print(
    f"Mean residual      : "
    f"{final_error_analysis['Residual'].mean():.5f}"
)

print(
    f"Median residual    : "
    f"{final_error_analysis['Residual'].median():.5f}"
)

print(
    f"Mean absolute log error: "
    f"{final_error_analysis['AbsLogError'].mean():.5f}"
)

print(
    f"Median absolute log error: "
    f"{final_error_analysis['AbsLogError'].median():.5f}"
)


print("\nTop 15 largest final ensemble OOF errors:")

display(
    final_error_analysis.nlargest(
        15,
        "AbsLogError"
    )[
        [
            "Id",
            "ActualPrice",
            "PredictedPrice",
            "PriceError",
            "PercentError",
            "Residual",
            "AbsLogError"
        ]
    ]
)

FINAL ENSEMBLE ERROR SUMMARY
----------------------------
Mean residual      : 0.00078
Median residual    : 0.00488
Mean absolute log error: 0.07463
Median absolute log error: 0.05223

Top 15 largest final ensemble OOF errors:


,Id,ActualPrice,PredictedPrice,PriceError,PercentError,Residual,AbsLogError
30,31,40000.0,85948.973788,45948.973788,114.872434,-0.764861,0.764861
631,633,82500.0,175032.593683,92532.593683,112.160720,-0.752168,0.752168
462,463,62383.0,123301.285664,60918.285664,97.652062,-0.681330,0.681330
1322,1325,147000.0,284582.297030,137582.297030,93.593399,-0.660587,0.660587
495,496,34900.0,67562.379225,32662.379225,93.588479,-0.660551,0.660551
88,89,85000.0,48477.839524,-36522.160476,-42.967248,0.561536,0.561536
969,971,135000.0,77104.249041,-57895.750959,-42.885741,0.560111,0.560111
915,917,35311.0,58568.818592,23257.818592,65.865647,-0.505997,0.505997
967,969,37900.0,62262.798404,24362.798404,64.281790,-0.496403,0.496403
410,411,60000.0,96853.924092,36853.924092,61.423207,-0.478853,0.478853


### Interpretation

The final Ridge-Optuna XGBoost ensemble shows very little systematic bias.
The mean residual is close to zero (**0.00078**), indicating that the model
does not consistently overpredict or underpredict sale prices across the
cleaned training set.

The median absolute log error (**0.05223**) is lower than the mean absolute
log error (**0.07463**), showing that most predictions are relatively accurate
while a smaller number of difficult observations produce substantially larger
errors.

The largest remaining OOF errors are concentrated in individual properties
rather than indicating a general failure of the ensemble. For example, some
low-priced houses are substantially overpredicted, while several higher-priced
properties are underpredicted.

Importantly, these observations are retained. Large OOF residuals alone are
not treated as sufficient evidence for further outlier removal. This avoids
iteratively deleting difficult observations simply because the selected model
predicts them poorly.

Overall, the residual analysis supports the selected 50/50 Ridge-Optuna
XGBoost ensemble: it achieves the best OOF RMSLE while maintaining little
systematic prediction bias.

## 26. Reliability by Price Range

The final OOF errors are grouped by sale-price segment.

This tests whether model reliability is uniform across the housing market. Large differences between price groups would indicate that the global RMSLE should not be interpreted as equal performance for every type of property.


In [37]:
# Cell 26 - Error analysis by price range

final_error_analysis["PriceRange"] = pd.qcut(
    final_error_analysis["ActualPrice"],
    q=4,
    labels=[
        "Low",
        "Lower-Middle",
        "Upper-Middle",
        "High"
    ]
)

price_range_analysis = (
    final_error_analysis
    .groupby("PriceRange", observed=True)
    .agg(
        Count=("Id", "count"),
        MeanActualPrice=("ActualPrice", "mean"),
        MeanPredictedPrice=("PredictedPrice", "mean"),
        MeanAbsLogError=("AbsLogError", "mean"),
        MedianAbsLogError=("AbsLogError", "median"),
        MeanPercentError=("PercentError", "mean")
    )
)

display(price_range_analysis)
print("\nRMSLE by price range:")

for price_range in final_error_analysis["PriceRange"].cat.categories:

    group = final_error_analysis[
        final_error_analysis["PriceRange"] == price_range
    ]

    group_rmsle = np.sqrt(
        np.mean(
            (
                group["PredictedLogPrice"]
                - group["ActualLogPrice"]
            ) ** 2
        )
    )

    print(
        f"{price_range:15s}: "
        f"{group_rmsle:.5f}"
    )


,Count,MeanActualPrice,MeanPredictedPrice,MeanAbsLogError,MedianAbsLogError,MeanPercentError
PriceRange,,,,,,
Low,365,105831.594521,109248.844801,0.098348,0.066425,4.808647
Lower-Middle,366,145001.016393,144121.996751,0.067526,0.047167,-0.569576
Upper-Middle,365,185263.052055,185223.314852,0.057846,0.045361,-0.017423
High,362,288619.552486,281272.011219,0.074820,0.058712,-2.016380



RMSLE by price range:
Low            : 0.14921
Lower-Middle   : 0.09951
Upper-Middle   : 0.07700
High           : 0.10051


### Interpretation

The price-range analysis shows that the final ensemble does not have uniform
predictive reliability across all segments of the housing market.

The **Upper-Middle** price segment achieves the lowest RMSLE (**0.07700**),
indicating that the ensemble performs particularly well for properties in
this part of the market. The **Lower-Middle** segment also shows relatively
strong performance with an RMSLE of **0.09951**.

The **Low** price segment is substantially more difficult, with an RMSLE of
**0.14921**, the highest among the four groups. Its mean percentage error is
also positive (**+4.81%**), suggesting a tendency to overpredict lower-priced
properties.

At the opposite end of the market, the **High** price segment has an RMSLE
of **0.10051** and a mean percentage error of approximately **-2.02%**,
indicating a mild tendency to underpredict expensive houses.

This pattern suggests a regression-to-the-mean effect: lower-priced houses
tend to be predicted somewhat higher, while expensive houses tend to be
predicted somewhat lower.

Therefore, the global OOF RMSLE of the final ensemble should not be
interpreted as uniform performance across all price levels. The model is most
reliable in the middle and upper-middle parts of the price distribution,
while low-priced properties remain the most challenging segment.

## 27. Reliability by Overall Quality

OOF performance is examined across `OverallQual`.

This helps determine whether the model behaves differently for low-quality, typical and premium houses, and whether rare quality groups produce less stable estimates.


In [38]:
# Cell 27 - Error analysis by OverallQual

# Add OverallQual from the clean training data
clean_train = train.loc[clean_mask].reset_index(drop=True)

final_error_analysis["OverallQual"] = clean_train["OverallQual"]

quality_analysis = (
    final_error_analysis
    .groupby("OverallQual")
    .agg(
        Count=("Id", "count"),
        MeanActualPrice=("ActualPrice", "mean"),
        MeanPredictedPrice=("PredictedPrice", "mean"),
        MeanAbsLogError=("AbsLogError", "mean"),
        MedianAbsLogError=("AbsLogError", "median"),
        MeanPercentError=("PercentError", "mean")
    )
)

display(quality_analysis)


# RMSLE for each quality group
print("\nRMSLE by OverallQual:")

for quality in sorted(
    final_error_analysis["OverallQual"].unique()
):
    group = final_error_analysis[
        final_error_analysis["OverallQual"] == quality
    ]

    group_rmsle = np.sqrt(
        np.mean(
            (
                group["PredictedLogPrice"]
                - group["ActualLogPrice"]
            ) ** 2
        )
    )

    print(
        f"OverallQual {quality:2d} "
        f"(n={len(group):3d}): "
        f"{group_rmsle:.5f}"
    )

,Count,MeanActualPrice,MeanPredictedPrice,MeanAbsLogError,MedianAbsLogError,MeanPercentError
OverallQual,,,,,,
1,2,50150.000000,56777.207608,0.234286,0.234286,19.146044
2,3,51770.333333,56439.718275,0.253271,0.215152,16.816514
3,20,87473.750000,84762.784255,0.150161,0.082630,0.898161
4,116,108420.655172,106055.288739,0.117062,0.087885,0.866322
5,397,133523.347607,132197.224126,0.074925,0.050217,0.118065
6,374,161603.034759,161384.456175,0.063058,0.050301,0.605947
7,319,207716.423197,207532.730137,0.065170,0.047824,0.819676
8,168,274735.535714,274329.526405,0.073067,0.052542,1.032446
9,43,367513.023256,354201.010011,0.072660,0.052150,-2.530688



RMSLE by OverallQual:
OverallQual  1 (n=  2): 0.27711
OverallQual  2 (n=  3): 0.31823
OverallQual  3 (n= 20): 0.21729
OverallQual  4 (n=116): 0.16878
OverallQual  5 (n=397): 0.10888
OverallQual  6 (n=374): 0.08514
OverallQual  7 (n=319): 0.09308
OverallQual  8 (n=168): 0.11106
OverallQual  9 (n= 43): 0.09703
OverallQual 10 (n= 16): 0.11381


### Interpretation

The OverallQual analysis shows that prediction reliability varies across
quality levels, but the group sizes must be considered when interpreting
the results.

The lowest quality categories have the largest RMSLE values:
**0.27711** for OverallQual 1 and **0.31823** for OverallQual 2.
However, these groups contain only **2** and **3** observations respectively,
so their error estimates are highly unstable and should not be treated as
evidence of a general model failure.

Among the well-represented quality groups, the ensemble performs particularly
well for typical-to-good houses. OverallQual 6 achieves an RMSLE of
**0.08514** across **374** observations, while OverallQual 7 achieves
**0.09308** across **319** observations.

Performance deteriorates for lower-quality houses. OverallQual 4 has an
RMSLE of **0.16878** across **116** observations, supporting the earlier
price-range analysis in which lower-priced properties were the most
difficult segment.

The highest-quality groups do not show the same severe degradation.
OverallQual 9 achieves an RMSLE of **0.09703**, while OverallQual 10 has
an RMSLE of **0.11381**. The latter contains only **16** observations,
so its estimate should again be interpreted cautiously.

Overall, the ensemble is most reliable for the densely represented
middle-quality properties. The apparent extreme errors at the lowest
quality levels are based on very small samples and therefore do not justify
additional model changes or observation removal.

## 28. Reliability by Neighborhood

Housing relationships may vary geographically, so OOF RMSLE is examined by neighborhood.

Neighborhood-level results with small sample sizes must be interpreted cautiously: a high or low score based on very few observations may not generalize.


In [39]:
# Cell 28 - Error analysis by Neighborhood

final_error_analysis["Neighborhood"] = clean_train["Neighborhood"]

neighborhood_analysis = (
    final_error_analysis
    .groupby("Neighborhood")
    .agg(
        Count=("Id", "count"),
        MeanActualPrice=("ActualPrice", "mean"),
        MeanPredictedPrice=("PredictedPrice", "mean"),
        MeanAbsLogError=("AbsLogError", "mean"),
        MedianAbsLogError=("AbsLogError", "median"),
        MeanPercentError=("PercentError", "mean")
    )
)

# Calculate RMSLE for each neighborhood
neighborhood_rmsle = {}

for neighborhood in final_error_analysis["Neighborhood"].unique():

    group = final_error_analysis[
        final_error_analysis["Neighborhood"] == neighborhood
    ]

    rmsle = np.sqrt(
        np.mean(
            (
                group["PredictedLogPrice"]
                - group["ActualLogPrice"]
            ) ** 2
        )
    )

    neighborhood_rmsle[neighborhood] = rmsle


neighborhood_analysis["RMSLE"] = pd.Series(
    neighborhood_rmsle
)

# Sort from worst to best RMSLE
neighborhood_analysis = neighborhood_analysis.sort_values(
    "RMSLE",
    ascending=False
)

display(neighborhood_analysis)

,Count,MeanActualPrice,MeanPredictedPrice,MeanAbsLogError,MedianAbsLogError,MeanPercentError,RMSLE
Neighborhood,,,,,,,
IDOTRR,37,100123.783784,98706.738997,0.163555,0.117658,5.517146,0.241499
OldTown,113,128225.300885,126479.634919,0.105492,0.073706,1.217989,0.143529
ClearCr,28,212565.428571,206794.347020,0.107002,0.095362,-1.423428,0.142972
SWISU,25,142591.360000,142429.054889,0.098617,0.078418,0.514112,0.129333
StoneBr,25,310499.000000,295549.187229,0.086556,0.062170,-3.504562,0.128326
BrkSide,58,124834.051724,121079.749141,0.093677,0.074845,-0.369375,0.126592
Edwards,98,127318.571429,125946.596382,0.093398,0.070956,0.876291,0.125504
Sawyer,74,136793.135135,136926.823084,0.074458,0.049740,1.380258,0.122179
NWAmes,73,189050.068493,191021.093870,0.067799,0.045621,1.732060,0.114888


### Interpretation

The neighborhood analysis confirms that the final ensemble does not perform
uniformly across locations.

`IDOTRR` is the most difficult neighborhood, with an OOF RMSLE of **0.24150**
across **37 observations**. `OldTown` also shows relatively high error
(**0.14353**) despite having **113 observations**, suggesting that the weaker
performance in these neighborhoods is not explained only by extremely small
sample sizes.

In contrast, `CollgCr` achieves an RMSLE of **0.06408** across **150
observations**, providing stronger evidence that the ensemble performs
reliably in this neighborhood. `Gilbert` (**0.07214**, n=79) and `SawyerW`
(**0.07093**, n=59) also show comparatively strong performance.

Some apparently excellent neighborhood scores must be interpreted cautiously.
For example, `Blueste` has an RMSLE of **0.04178**, but contains only **2
observations**, while `NPkVill` has an RMSLE of **0.04632** across only **9
observations**. These small samples are insufficient for strong conclusions
about generalization.

The direction of the percentage errors also varies geographically. For
example, `StoneBr` shows a mean percentage error of approximately **-3.50%**,
indicating some underprediction, whereas `IDOTRR` shows approximately
**+5.52%**, indicating average overprediction.

Together with the price-range and OverallQual analyses, these results show
that the global OOF RMSLE of **0.10978** hides meaningful subgroup variation.
The ensemble is strongest for well-represented market segments such as
`CollgCr`, while lower-priced and certain older neighborhoods remain more
challenging.

These subgroup results are treated as diagnostic evidence rather than as a
basis for additional tuning or observation removal.

## 29. Final Training

Only after model selection, hyperparameter optimization and error analysis
are complete are the selected pipelines fitted on all **clean training
observations**.

The final ensemble consists of the leakage-free Ridge model and the
**Optuna-tuned XGBoost model**, combined with equal weights as selected from
OOF predictions.

The final XGBoost configuration uses the hyperparameters selected by Optuna
under the same five-fold leakage-free validation protocol.

The Kaggle test set is transformed only through preprocessors fitted on the
clean training data. It never participates in model fitting, hyperparameter
selection, preprocessing estimation, or ensemble-weight selection.

The final models are therefore trained on all **1,458 clean training
observations** only after all model-selection decisions have been completed.

In [42]:
# Cell 29 - Final training on all clean training data

# Final Ridge pipeline
final_ridge = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", Ridge(alpha=10.0))
])

# Final Optuna-tuned XGBoost pipeline
final_xgb = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", xgb.XGBRegressor(
        **study.best_params,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

print("Training final Ridge...")
final_ridge.fit(X_clean, y_clean)

print("Training final Optuna-tuned XGBoost...")
final_xgb.fit(X_clean, y_clean)

print("\nFinal models trained successfully.")
print("Training observations:", len(X_clean))
print("Test observations    :", len(X_test_fe))
print("Final ensemble       : 50% Ridge + 50% Optuna XGBoost")

print("\nSelected Optuna XGBoost parameters:")
for key, value in study.best_params.items():
    print(f"{key:20s}: {value}")

Training final Ridge...
Training final Optuna-tuned XGBoost...

Final models trained successfully.
Training observations: 1458
Test observations    : 1459
Final ensemble       : 50% Ridge + 50% Optuna XGBoost

Selected Optuna XGBoost parameters:
n_estimators        : 1100
learning_rate       : 0.039524534394525634
max_depth           : 3
min_child_weight    : 3
subsample           : 0.7750590251178121
colsample_bytree    : 0.6515493505666603
reg_alpha           : 0.1030285029296229
reg_lambda          : 0.6672670420130714


### Interpretation

The final Ridge and Optuna-tuned XGBoost pipelines were successfully fitted
on all 1,458 clean training observations.

The selected 50/50 ensemble is now ready to generate predictions for the
untouched Kaggle test set. No test-set information was used during
preprocessing, model fitting, hyperparameter optimization, or ensemble
selection.

## 30. Kaggle Test Predictions and Submission

Predictions from the selected final models are combined using the chosen ensemble weights.

Because the models predict log-prices, the ensemble output is transformed back to the original price scale with `np.expm1()` before the Kaggle submission file is created.


In [44]:
# Cell 30 - Final test predictions and Kaggle submission

# Predict log(SalePrice) with both final models
ridge_test_pred_log = final_ridge.predict(X_test_fe)
xgb_test_pred_log = final_xgb.predict(X_test_fe)

# Final 50/50 ensemble in log space
final_test_pred_log = (
    0.5 * ridge_test_pred_log
    + 0.5 * xgb_test_pred_log
)

# Convert back to original SalePrice scale
final_test_pred = np.expm1(final_test_pred_log)

# Safety checks
print("Prediction checks")
print("-----------------")
print("Number of predictions :", len(final_test_pred))
print("NaN predictions       :", np.isnan(final_test_pred).sum())
print("Infinite predictions  :", np.isinf(final_test_pred).sum())
print("Negative predictions  :", (final_test_pred < 0).sum())

print("\nPrediction summary:")
print(pd.Series(final_test_pred).describe())

# Create Kaggle submission
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": final_test_pred
})

# Final validation
assert len(submission) == len(test)
assert submission["SalePrice"].notna().all()
assert np.isfinite(submission["SalePrice"]).all()
assert (submission["SalePrice"] > 0).all()

# Save to submissions folder
submission_path = (
    "submissions/"
    "submission_leakage_free_ridge_optuna_xgb_50_50.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print("\nSubmission created successfully:")
print(submission_path)

display(submission.head(10))

Prediction checks
-----------------
Number of predictions : 1459
NaN predictions       : 0
Infinite predictions  : 0
Negative predictions  : 0

Prediction summary:
count    1.459000e+03
mean     1.774072e+05
std      8.076537e+04
min      4.248742e+04
25%      1.256818e+05
50%      1.556876e+05
75%      2.071632e+05
max      1.052211e+06
dtype: float64

Submission created successfully:
submissions/submission_leakage_free_ridge_optuna_xgb_50_50.csv


,Id,SalePrice
0,1461,119326.512714
1,1462,157566.349817
2,1463,180707.377936
3,1464,198034.729260
4,1465,184622.543404
5,1466,171422.090907
6,1467,179387.073353
7,1468,164870.138494
8,1469,186304.224699
9,1470,122623.373863


### Interpretation

The final submission contains predictions for all **1,459 Kaggle test
observations**, with no missing, infinite, or negative predicted prices.

Predictions are generated independently by the final Ridge and
Optuna-tuned XGBoost pipelines and combined with the **50/50 ensemble
weights selected from leakage-free OOF validation**.

The ensemble is performed in log-price space before applying the inverse
`expm1()` transformation. This is consistent with the target transformation
and the RMSLE-based evaluation framework used throughout the project.

The Kaggle test set was used only for final inference and did not influence
feature preprocessing, hyperparameter optimization, model selection, or
ensemble-weight selection.

## 31. Prediction Sanity Checks

Before submission, predictions are checked for:

- the expected number of rows;
- missing values;
- infinite values;
- negative prices;
- implausible extremes.

These checks do not prove that predictions are correct, but they catch common pipeline and transformation failures before deployment or submission.


In [45]:
# Cell 31 - Final prediction sanity check

prediction_check = X_test_fe.copy()

prediction_check["Id"] = test["Id"].values
prediction_check["PredictedSalePrice"] = final_test_pred

check_columns = [
    "Id",
    "Neighborhood",
    "OverallQual",
    "GrLivArea",
    "TotalSF",
    "YearBuilt",
    "GarageCars",
    "PredictedSalePrice"
]

print("10 highest predicted prices:")

display(
    prediction_check
    .nlargest(10, "PredictedSalePrice")[check_columns]
)

print("\n10 lowest predicted prices:")

display(
    prediction_check
    .nsmallest(10, "PredictedSalePrice")[check_columns]
)

10 highest predicted prices:


,Id,Neighborhood,OverallQual,GrLivArea,TotalSF,YearBuilt,GarageCars,PredictedSalePrice
1089,2550,Edwards,10,5095,10190.0,2008,3.0,1.052211e+06
1222,2683,NoRidge,9,3500,5233.0,1993,3.0,5.769766e+05
203,1664,NridgHt,10,2674,5304.0,2007,3.0,5.462382e+05
19,1480,NridgHt,9,2696,5542.0,2003,3.0,5.346102e+05
803,2264,StoneBr,9,2338,4998.0,2006,3.0,5.171000e+05
1168,2629,StoneBr,10,3390,4918.0,2006,3.0,5.099137e+05
834,2295,NridgHt,10,2290,4610.0,2007,3.0,5.075680e+05
832,2293,NridgHt,9,2552,5104.0,2007,3.0,5.057717e+05
217,1678,NridgHt,10,2492,4984.0,2004,3.0,4.999188e+05
1167,2628,StoneBr,9,2798,4196.0,2005,3.0,4.990792e+05



10 lowest predicted prices:


,Id,Neighborhood,OverallQual,GrLivArea,TotalSF,YearBuilt,GarageCars,PredictedSalePrice
1431,2892,IDOTRR,3,729,729.0,1945,0.0,42487.424084
362,1823,IDOTRR,3,797,1042.0,1900,0.0,42793.328115
1433,2894,IDOTRR,3,936,1152.0,1916,0.0,47428.789890
1331,2792,IDOTRR,3,1020,2040.0,1918,0.0,53034.245887
453,1914,IDOTRR,4,572,1144.0,1925,1.0,55005.170448
1411,2872,Edwards,2,498,996.0,1922,1.0,55120.323397
1428,2889,IDOTRR,4,672,1104.0,1925,0.0,56358.685547
354,1815,OldTown,2,612,612.0,1940,1.0,56495.609425
359,1820,OldTown,4,816,1440.0,1910,0.0,56825.438882
638,2099,OldTown,2,407,814.0,1946,1.0,57630.143752


### Interpretation

The prediction sanity check does not reveal an obvious inference or
transformation failure.

The highest predicted price is approximately **$1.05 million** for an
unusually large, high-quality property with `OverallQual = 10`,
`GrLivArea = 5,095` and `TotalSF = 10,190`. Although this prediction is
substantially higher than the remaining test predictions, the underlying
property is also structurally extreme. Therefore, no arbitrary clipping is
applied without validation evidence that such a correction would improve
generalization.

The lowest predictions are concentrated among small, older and lower-quality
properties, particularly in neighborhoods such as `IDOTRR` and `OldTown`.
This pattern is consistent with the subgroup reliability analysis performed
on the OOF predictions.

Overall, the extreme test predictions remain directionally consistent with
the corresponding property characteristics. The sanity check therefore
provides no evidence of NaN propagation, inverse-transformation errors, or
obviously invalid final predictions.

## 32. OOF vs Kaggle Generalization

The final model is evaluated on the Kaggle Public Leaderboard only after
model selection has been completed using leakage-free OOF validation.

The OOF-to-Public generalization gap is defined as

$$
\Delta =
\mathrm{RMSLE}_{Kaggle}
-
\mathrm{RMSLE}_{OOF}.
$$

A positive value of $\Delta$ means that the internal cross-validation
estimate was optimistic relative to the Kaggle Public Leaderboard.

The selected 50/50 Ridge + Optuna-tuned XGBoost ensemble achieved:

- **OOF RMSLE:** 0.10978
- **Kaggle Public Score:** 0.12789
- **OOF-to-Public gap:** approximately 0.01811

For comparison, an earlier 65% Ridge + 35% XGBoost submission achieved a
slightly weaker OOF RMSLE of **0.10987**, but a better Kaggle Public Score
of **0.12575**, corresponding to a smaller generalization gap of
approximately **0.01588**.

This is an important generalization result. The configuration with the best
internal OOF score is not necessarily the configuration with the best score
on a particular external leaderboard subset. The difference may reflect
sampling variation between the cross-validation folds and the public
leaderboard subset.

The earlier 65/35 result is reported for transparency, but it is not used
retroactively to change the final model. Hyperparameters and ensemble weights
remain selected from the leakage-free OOF validation procedure rather than
from repeated Public Leaderboard feedback.

Repeatedly tuning the pipeline to the Public Leaderboard would risk
leaderboard overfitting. Therefore, the Kaggle score is treated as an
external generalization check rather than as another validation fold.

In [47]:
# Cell 32 - OOF vs Kaggle generalization comparison

final_oof_score = blend_oof_rmsle
final_public_score = 0.12789

previous_oof_score = 0.10987
previous_public_score = 0.12575

final_gap = final_public_score - final_oof_score
previous_gap = previous_public_score - previous_oof_score

print("FINAL GENERALIZATION CHECK")
print("--------------------------")

print("\nSelected final model:")
print(f"OOF RMSLE           : {final_oof_score:.5f}")
print(f"Kaggle Public Score : {final_public_score:.5f}")
print(f"Generalization Gap  : {final_gap:.5f}")

print("\nEarlier 65/35 ensemble:")
print(f"OOF RMSLE           : {previous_oof_score:.5f}")
print(f"Kaggle Public Score : {previous_public_score:.5f}")
print(f"Generalization Gap  : {previous_gap:.5f}")

print("\nComparison:")
print(
    f"OOF advantage of final model   : "
    f"{previous_oof_score - final_oof_score:+.5f}"
)

print(
    f"Public advantage of 65/35 model: "
    f"{final_public_score - previous_public_score:+.5f}"
)

FINAL GENERALIZATION CHECK
--------------------------

Selected final model:
OOF RMSLE           : 0.10978
Kaggle Public Score : 0.12789
Generalization Gap  : 0.01811

Earlier 65/35 ensemble:
OOF RMSLE           : 0.10987
Kaggle Public Score : 0.12575
Generalization Gap  : 0.01588

Comparison:
OOF advantage of final model   : +0.00009
Public advantage of 65/35 model: +0.00214


### Interpretation

The final 50/50 Ridge + Optuna-tuned XGBoost ensemble achieves the best
internal OOF RMSLE of **0.10978**, improving slightly on the earlier 65/35
ensemble OOF score of **0.10987**.

However, the earlier 65/35 ensemble achieves the better Kaggle Public Score:
**0.12575** compared with **0.12789** for the selected final model. Its
OOF-to-Public generalization gap is also smaller (**0.01588** versus
**0.01811**).

The OOF advantage of the final model is only **0.00009**, whereas the earlier
65/35 ensemble has a **0.00214** advantage on the Public Leaderboard. This
demonstrates that a marginal improvement in cross-validation does not
guarantee an improvement on an external evaluation subset.

This discrepancy is treated as evidence about the uncertainty of the
validation estimate rather than as a reason for additional leaderboard-driven
tuning. The final model remains selected according to the predefined
leakage-free OOF procedure, while the stronger earlier Kaggle result is
reported transparently.

The Public Leaderboard is therefore used as an external generalization check,
not as an additional validation fold or a source of hyperparameter and
ensemble-weight decisions.

## 33. Final Conclusions and Limitations

The final model is selected through a reproducible leakage-free workflow rather than by repeatedly searching for the best public leaderboard score.

### Main findings

1. **Leakage-free preprocessing is essential.** Imputation, scaling and categorical encoding are fitted only inside each training fold.
2. **True OOF predictions make error analysis possible.** Every training observation is evaluated by a model that did not train on it.
3. **Structural outlier analysis improved understanding and stability.** The two removed observations are identified by an explicit property-based rule rather than by prediction error alone.
4. **Different model families provide complementary information**. Ridge captures regularized linear structure, while the Optuna-tuned XGBoost captures nonlinear interactions.
5. **Ensembling is retained only when OOF evidence supports it.**
6. **More models are not necessarily better.** LightGBM is tested, but additional complexity is rejected when it does not improve the selected ensemble.
7. **Model reliability varies across market segments.** Price range, quality and neighborhood analyses show that global RMSLE does not describe every subgroup equally well.
8. **The Kaggle score is an external check**. The final model achieved the best OOF RMSLE, while an earlier 65/35 ensemble achieved a better Public Score. This discrepancy is documented rather than used for leaderboard-driven retuning.

### Limitations

- The dataset is relatively small, so subgroup estimates can be noisy.
- A single five-fold split still has sampling variability.
- The structural outlier rule is specific to this dataset and should be revalidated before use on another housing market.
- The public leaderboard represents only part of the hidden Kaggle test labels.
- A production system would require monitoring for distribution drift, changing market conditions and subgroup-specific degradation.

The project should therefore be interpreted as a validated machine-learning experiment with documented assumptions and limitations, not only as a Kaggle score.


## 34. Final model-selection logic

The project follows a fixed decision sequence:

1. establish a leakage-free OOF baseline;
2. inspect large OOF errors and define any structural outlier rule transparently;
3. retrain and compare Ridge and XGBoost under identical folds;
4. test whether prediction diversity improves an OOF ensemble;
5. evaluate LightGBM as a controlled negative experiment rather than assuming that another complex model must help;
6. inspect errors by price range, quality and neighborhood;
7. sanity-check extreme test predictions instead of clipping them automatically;
8. compare OOF performance with the Kaggle public score only for the **same submitted model**.

The final model should therefore be selected from evidence, not from the number of algorithms tried. Any Kaggle Public Score reported in this notebook corresponds to a clearly identified submitted model and is treated only as an external evaluation result.
